# Step 3 — Skip-gram (SGNS) on Walk Sentences, four-method comparison

Train one Skip-gram model per filter method and export each one's top-10 nearest-neighbour recommendations. The four methods each produced their own filtered walk sentences upstream (`filters/{method}/{value}/walk_sentences.jsonl`); this notebook feeds each corpus through the **identical** Word2Vec training, so any difference in results is attributable to the input corpus (which filter method produced it), never to a training setting that quietly differed between runs. One shared mechanism, only the input changes.

The department-restricted disparity variant is deliberately **excluded** — it was a negative result, not a candidate for the four-way comparison.

**Inputs** (in `outputs/`)
- `filters/{method}/{value}/walk_sentences.jsonl` — one per method: constant_theta/theta_0.70, pct_dropoff/drop_0.10, kneedle/sensitivity_1.0, disparity/alpha_0.05
- `shared_sample.json` — the walk's own starting products, for the evaluation sample
- `embeddings.npy`, `product_id_to_index.json`, `products_text.csv` (Step 1, validation only)

**Outputs** (to `outputs/`, one set per method)
- `skipgram/{method}/{value}/model.gensim`, `skipgram/{method}/{value}/embeddings.npy`, `skipgram/{method}/{value}/product_id_to_index.json`, `skipgram/{method}/{value}/config.json`
- `skipgram/{method}/{value}/top10.jsonl` — top-10 neighbours for every product in that method's vocabulary
- `evaluation_sample.json` — one shared evaluation anchor set (built once, used by all four and the eval notebook)

## 1. Imports and paths

Paths are resolved relative to the project root so the notebook works regardless of where Jupyter is launched from.

In [1]:
import json
import datetime
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from gensim.models import Word2Vec

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
STEP1_DIR = OUT_DIR / "step1"
FILTERS_DIR = OUT_DIR / "filters"
SKIPGRAM_DIR = OUT_DIR / "skipgram"
SKIPGRAM_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:   ", DATA_DIR)
print("Outputs dir:", OUT_DIR)

Data dir:    /Users/linnhtet/Developer/bachelor-thesis/data
Outputs dir: /Users/linnhtet/Developer/bachelor-thesis/outputs


## 2. Parameters

Every training parameter is **identical across all four methods** (copied from the reference run) — the whole point of the comparison is that only the input corpus changes. `METHODS` lists the four filter corpora being compared.

In [2]:
# ---- Word2Vec / SGNS parameters — IDENTICAL across all four methods ----
# SG = 0 : CBOW, Continuous Bag Of Words - predicts the target from its context
# SG = 1 : reverse - predicts surrounding context tokens from a single target token
SG = 1                # 1 = Skip-gram (not CBOW), per Item2Vec
VECTOR_SIZE = 128     # embedding dimensionality (~49K catalog; sweepable)
WINDOW = 10           # >= max sentence length -> every in-sentence pair is a positive
MIN_COUNT = 1         # CRITICAL: keep every product; gensim default 5 drops sparse anchors
NEGATIVE = 15         # negative samples per positive (Item2Vec §4.2)
SAMPLE = 1e-3         # frequent-token subsampling — the popularity-confound fix
EPOCHS = 20           # Item2Vec §4.2
WORKERS = 4           # parallelism (fully reproducible only with WORKERS=1 + fixed SEED)
SEED = 42             # reproducibility

# The four filter methods being compared. The department-restricted disparity
# variant is deliberately excluded (a negative result, not a candidate).
METHODS = ["constant_theta", "pct_dropoff", "kneedle", "disparity"]

# Each method maps to its value-labeled subfolder under filters/ and skipgram/.
METHOD_DIRS = {
    "constant_theta": "constant_theta/theta_0.70",
    "pct_dropoff": "pct_dropoff/drop_0.10",
    "kneedle": "kneedle/sensitivity_1.0",
    "disparity": "disparity/alpha_0.05",
}

# ---- Stress-test anchors — same product_ids as Steps 1-2 ----
STRESS_TEST_IDS = {
    4210:  "Whole Milk",
    6347:  "Unsweetened Almond Milk",
    651:   "Organic Salted Butter",
    16268: "Vegan Butter",
    6104:  "Whole Milk Plain Yogurt",
    920:   "Coconut Yogurt",
}

## 3. Train one model per method

For each method: load its filtered walk sentences, build the corpus exactly as the reference does (one sentence per walk record, duplicates and order kept as-is), print the same corpus diagnostics (useful side-by-side comparison data on its own), train with the identical parameters above, and save the model + vectors + id map + config. Per-method results are kept in memory for the evaluation sample (§4), the export (§5), and validation (§6).

In [3]:
def build_corpus(method):
    """Load a method's kept sentences: one sentence per walk record, duplicates and
    order kept exactly as-is (same as the reference)."""
    records = []
    with open(FILTERS_DIR / METHOD_DIRS[method] / "walk_sentences.jsonl") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return [[str(pid) for pid in rec["sentence"]] for rec in records]


results = {}   # method -> {tokens, sg_pid_to_idx, p3_emb, p3_idx_to_pid, vocab_size, config}

for method in METHODS:
    print("=" * 78)
    print(f"METHOD: {method}")
    print("=" * 78)

    corpus = build_corpus(method)

    # ---- Corpus diagnostics (same as the reference) ----
    lengths = [len(s) for s in corpus]
    token_counts = Counter(tok for s in corpus for tok in s)
    total_tokens = sum(lengths)
    print(f"training sentences     : {len(corpus):,}")
    print(f"sentence length min/mean/median/max: "
          f"{min(lengths)} / {np.mean(lengths):.2f} / {int(np.median(lengths))} / {max(lengths)}")
    print(f"distinct product tokens: {len(token_counts):,}")
    print(f"total tokens           : {total_tokens:,}")

    # ---- Train (identical parameters for every method) ----
    model = Word2Vec(
        sentences=corpus,
        sg=SG,
        vector_size=VECTOR_SIZE,
        window=WINDOW,
        min_count=MIN_COUNT,
        negative=NEGATIVE,
        sample=SAMPLE,
        epochs=EPOCHS,
        workers=WORKERS,
        seed=SEED,
    )
    print(f"vocabulary (products with a vector): {len(model.wv):,}")
    print(f"embedding matrix shape             : {model.wv.vectors.shape}")

    # ---- Save per-method artifacts ----
    (SKIPGRAM_DIR / METHOD_DIRS[method]).mkdir(parents=True, exist_ok=True)
    tokens = list(model.wv.index_to_key)
    sg_embeddings = model.wv.vectors                    # (vocab, VECTOR_SIZE), raw (unnormalised)
    sg_pid_to_idx = {int(tok): i for i, tok in enumerate(tokens)}

    model.save(str(SKIPGRAM_DIR / METHOD_DIRS[method] / "model.gensim"))
    np.save(SKIPGRAM_DIR / METHOD_DIRS[method] / "embeddings.npy", sg_embeddings)
    with open(SKIPGRAM_DIR / METHOD_DIRS[method] / "product_id_to_index.json", "w") as f:
        json.dump({str(pid): i for pid, i in sg_pid_to_idx.items()}, f)

    config = {
        "method": method,
        "sg": SG,
        "vector_size": VECTOR_SIZE,
        "window": WINDOW,
        "min_count": MIN_COUNT,
        "negative": NEGATIVE,
        "sample": SAMPLE,
        "epochs": EPOCHS,
        "workers": WORKERS,
        "seed": SEED,
        "n_sentences": len(corpus),
        "n_distinct_tokens": len(token_counts),
        "total_tokens": total_tokens,
        "vocab_size": len(model.wv),
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    }
    with open(SKIPGRAM_DIR / METHOD_DIRS[method] / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    # Unit-normalised Step 3 vectors for cosine (used by the export and validation).
    p3_emb = sg_embeddings / np.clip(np.linalg.norm(sg_embeddings, axis=1, keepdims=True), 1e-12, None)

    results[method] = {
        "tokens": tokens,
        "sg_pid_to_idx": sg_pid_to_idx,
        "p3_emb": p3_emb,
        "p3_idx_to_pid": {i: int(tok) for i, tok in enumerate(tokens)},
        "vocab_size": len(model.wv),
        "config": config,
    }
    print(f"Saved skipgram/{METHOD_DIRS[method]}/ "
          f"(model.gensim, embeddings.npy, product_id_to_index.json, config.json)")

print("\nAll four models trained.")

METHOD: constant_theta
training sentences     : 131,000
sentence length min/mean/median/max: 2 / 2.78 / 2 / 14
distinct product tokens: 30,117
total tokens           : 363,649


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


vocabulary (products with a vector): 30,117
embedding matrix shape             : (30117, 128)
Saved skipgram_model__constant_theta.gensim, skipgram_embeddings__constant_theta.npy, skipgram_product_id_to_index__constant_theta.json, skipgram_config__constant_theta.json
METHOD: pct_dropoff
training sentences     : 75,605
sentence length min/mean/median/max: 2 / 2.28 / 2 / 13
distinct product tokens: 26,814
total tokens           : 172,099
vocabulary (products with a vector): 26,814
embedding matrix shape             : (26814, 128)
Saved skipgram_model__pct_dropoff.gensim, skipgram_embeddings__pct_dropoff.npy, skipgram_product_id_to_index__pct_dropoff.json, skipgram_config__pct_dropoff.json
METHOD: kneedle
training sentences     : 167,568
sentence length min/mean/median/max: 2 / 2.68 / 2 / 16
distinct product tokens: 35,458
total tokens           : 449,902


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


vocabulary (products with a vector): 35,458
embedding matrix shape             : (35458, 128)
Saved skipgram_model__kneedle.gensim, skipgram_embeddings__kneedle.npy, skipgram_product_id_to_index__kneedle.json, skipgram_config__kneedle.json
METHOD: disparity
training sentences     : 104,771
sentence length min/mean/median/max: 2 / 2.67 / 2 / 16
distinct product tokens: 29,014
total tokens           : 279,711


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


vocabulary (products with a vector): 29,014
embedding matrix shape             : (29014, 128)
Saved skipgram_model__disparity.gensim, skipgram_embeddings__disparity.npy, skipgram_product_id_to_index__disparity.json, skipgram_config__disparity.json

All four models trained.


## 4. Build the shared evaluation sample (once)

Drawn from the walk's **own starting products** (`shared_sample.json`) — the anchors the walk was built to find substitutes for — not arbitrary vocabulary tokens that only ever appeared as someone else's walk winner. We draw 150 with a fixed seed, always add the six stress-test anchors, then record which of them have a trained vector under each method.

A method's vocabulary only contains products that appeared in at least one of its **own** kept sentences, so an anchor can have a vector under three methods but not the fourth if that method's filtering was strict enough to yield zero sentences for it (this happened to Whole Milk under the base disparity run). Per-method coverage is itself informative and kept visible; `all_four_coverage` is the fair, apples-to-apples set the evaluation notebook draws from.

In [4]:
# The walk's own starting products (not arbitrary vocabulary tokens).
with open(OUT_DIR / "shared_sample.json") as f:
    shared_sample = json.load(f)
starting_products = [int(p) for p in shared_sample["starting_products"]]

# Draw 150 anchors, then always include the six stress-test anchors (same pattern
# as the walk's own starting-product selection).
eval_rng = np.random.default_rng(42)
pool_size = min(150, len(starting_products))
candidate_pool = [int(x) for x in eval_rng.choice(np.array(starting_products),
                                                  size=pool_size, replace=False)]
for pid in STRESS_TEST_IDS:
    if pid not in candidate_pool:
        candidate_pool.append(pid)

# Per-method coverage: which of the pool have a trained vector under each method.
print("Coverage of the evaluation pool, per method:")
print("=" * 60)
coverage_by_method = {}
for method in METHODS:
    sg_pid_to_idx = results[method]["sg_pid_to_idx"]
    covered = [pid for pid in candidate_pool if pid in sg_pid_to_idx]
    coverage_by_method[method] = covered
    print(f"  {method:<16} {len(covered):>4}/{len(candidate_pool):<4} "
          f"({100 * len(covered) / max(len(candidate_pool), 1):.1f}%)")

# Anchors with a vector under ALL four methods: the fair comparison set.
all_four_coverage = [pid for pid in candidate_pool
                     if all(pid in results[m]["sg_pid_to_idx"] for m in METHODS)]
print(f"\n  all-four coverage (fair comparison set): "
      f"{len(all_four_coverage)}/{len(candidate_pool)}")

evaluation_sample = {
    "candidate_pool": candidate_pool,
    "coverage_by_method": coverage_by_method,
    "all_four_coverage": all_four_coverage,
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
}
with open(SKIPGRAM_DIR / "evaluation_sample.json", "w") as f:
    json.dump(evaluation_sample, f, indent=2)
print("Saved evaluation_sample.json")

Coverage of the evaluation pool, per method:
  constant_theta    128/156  (82.1%)
  pct_dropoff       128/156  (82.1%)
  kneedle           149/156  (95.5%)
  disparity         120/156  (76.9%)

  all-four coverage (fair comparison set): 106/156
Saved evaluation_sample.json


## 5. Export top-10 for every vocabulary token, per method

Mirrors the reference's export: for each method, compute every vocabulary product's top-10 by cosine similarity (excluding itself) and write `skipgram/{method}/{value}/top10.jsonl`. This covers the method's **full** vocabulary, not just the 150-anchor evaluation pool, so the tables stay reusable beyond this one comparison.

In [5]:
def p3_top_k(method, pid, k=10):
    """Top-k Step 3 neighbours of pid under a given method (excluding itself), as
    (pid, cosine). Returns None if pid has no vector under that method."""
    r = results[method]
    sg_pid_to_idx = r["sg_pid_to_idx"]
    if pid not in sg_pid_to_idx:
        return None
    p3_emb = r["p3_emb"]
    p3_idx_to_pid = r["p3_idx_to_pid"]
    sims = p3_emb @ p3_emb[sg_pid_to_idx[pid]]
    out = []
    for i in np.argsort(-sims):
        npid = p3_idx_to_pid[int(i)]
        if npid == pid:
            continue
        out.append((npid, float(sims[i])))
        if len(out) == k:
            break
    return out


for method in METHODS:
    tokens = results[method]["tokens"]
    anchor_ids = [int(tok) for tok in tokens]           # every product with a vector under this method
    (SKIPGRAM_DIR / METHOD_DIRS[method]).mkdir(parents=True, exist_ok=True)
    path = SKIPGRAM_DIR / METHOD_DIRS[method] / "top10.jsonl"
    with open(path, "w") as f:
        for pid in anchor_ids:
            neighbours = [npid for npid, _ in p3_top_k(method, pid, 10)]
            f.write(json.dumps({"anchor_id": pid, "neighbours": neighbours}) + "\n")
    print(f"  {method:<16} exported top-10 for {len(anchor_ids):,} anchors -> {path.name}")

  constant_theta   exported top-10 for 30,117 anchors -> skipgram_top10__constant_theta.jsonl
  pct_dropoff      exported top-10 for 26,814 anchors -> skipgram_top10__pct_dropoff.jsonl
  kneedle          exported top-10 for 35,458 anchors -> skipgram_top10__kneedle.jsonl
  disparity        exported top-10 for 29,014 anchors -> skipgram_top10__disparity.jsonl


## 6. Validation — four methods side by side

Comparison-first, the same pattern used throughout this project: group by anchor, then show all four methods for that anchor consecutively (not method-first). The rank-novelty check reuses the reference's `p1_ranks` logic verbatim, now applied to the trained Step 3 embeddings rather than the raw walk output.

In [6]:
# ---- Step 1 artifacts (validation only) ----
p1_emb = np.load(STEP1_DIR / "embeddings.npy").astype(np.float32)
p1_emb = p1_emb / np.clip(np.linalg.norm(p1_emb, axis=1, keepdims=True), 1e-12, None)
with open(STEP1_DIR / "product_id_to_index.json") as f:
    p1_pid_to_idx = {int(pid): row for pid, row in json.load(f).items()}
p1_idx_to_pid = {row: pid for pid, row in p1_pid_to_idx.items()}

# ---- Product names ----
products_text = pd.read_csv(STEP1_DIR / "products_text.csv")
id_to_name = dict(zip(products_text["product_id"], products_text["product_name"]))


def name(pid):
    return id_to_name.get(int(pid), f"?{pid}")


def p1_ranks(pid):
    """Step 1 rank lookup for an anchor (reuses Step 2's rank-novelty logic):
    ranks[row] with 0 = the anchor itself, 1 = its nearest neighbour."""
    sims = p1_emb @ p1_emb[p1_pid_to_idx[pid]]
    ranks = np.empty(len(sims), dtype=int)
    ranks[np.argsort(-sims)] = np.arange(len(sims))
    return sims, ranks

### 6.1 Vocabulary size per method

How many products got a trained vector at all under each method.

In [7]:
print("Vocabulary size per method (products with a trained vector):")
print("=" * 60)
for method in METHODS:
    print(f"  {method:<16} {results[method]['vocab_size']:,}")

Vocabulary size per method (products with a trained vector):
  constant_theta   30,117
  pct_dropoff      26,814
  kneedle          35,458
  disparity        29,014


### 6.2 Stress-test neighbours, all four methods per anchor

Each stress-test anchor's top-10 Step 3 neighbours, all four methods printed together under one anchor block for direct comparison.

In [8]:
for pid, nm in STRESS_TEST_IDS.items():
    print("\n" + "=" * 90)
    print(f"{nm} ({pid}) — top-10 neighbours by method")
    print("=" * 90)
    for method in METHODS:
        print(f"\n[{method}]")
        nbrs = p3_top_k(method, pid, 10)
        if nbrs is None:
            print("  no vector under this method (never appeared in a kept sentence)")
            continue
        for npid, sim in nbrs:
            print(f"  {sim:.3f}  {name(npid)}")


Whole Milk (4210) — top-10 neighbours by method

[constant_theta]
  0.804  Fat Free Skim Milk
  0.791  Skim Milk
  0.789  Milk, Fat Free
  0.781  Goat Milk
  0.763  Lowfat Goat Milk
  0.758  Milk, Organic, Fat Free
  0.753  Organic Low Fat Milk
  0.753  Milk Whole
  0.752  Vitamin D Organic Whole Milk
  0.751  Chocolate Lowfat Milk

[pct_dropoff]
  0.992  Chocolate Lowfat Milk
  0.991  Vitamin D Whole Milk
  0.991  1% Lowfat Milk
  0.991  Vitamin D Organic Whole Milk
  0.990  Lowfat Milk
  0.990  Milk, Organic, Vitamin D
  0.990  Organic Half & Half
  0.989  Low Fat Milk
  0.989  Fat Free Half & Half Ultra-Pasteurized
  0.989  Organic 4% Milk Fat Whole Milk Cottage Cheese

[kneedle]
  0.898  Fat Free Milk
  0.889  Dairy Pure Whole Milk
  0.881  Milk Whole
  0.876  Original Egg Substitute
  0.870  Skim Milk
  0.870  Milk, Organic, Vitamin D
  0.867  Chocolate Milk
  0.865  Fat Free Skim Milk
  0.863  Vitamin D Whole Milk
  0.863  Brown Eggs

[disparity]
  0.913  Milk
  0.913  Milk, Fat

### 6.3 Rank novelty — where each method's top-10 sits in Step 1's own ranking

For the same six anchors, all four methods side by side: for each neighbour the method surfaced, where does it sit in Step 1's own similarity ranking for that anchor? A neighbour ranked deep in Step 1 (rank > 10) is genuinely new signal the trained space found; one inside Step 1's top-10 was already obvious from text alone.

In [9]:
for pid, nm in STRESS_TEST_IDS.items():
    print("\n" + "=" * 90)
    print(f"{nm} ({pid}) — where each method's top-10 sits in Step 1's own ranking")
    print("=" * 90)
    if pid not in p1_pid_to_idx:
        print("  not in Step 1 catalogue, skipped")
        continue
    p1_sims, p1_rank = p1_ranks(pid)
    for method in METHODS:
        print(f"\n[{method}]")
        nbrs = p3_top_k(method, pid, 10)
        if nbrs is None:
            print("  no vector under this method (never appeared in a kept sentence)")
            continue
        for npid, sim in nbrs:
            if npid in p1_pid_to_idx:
                r = int(p1_rank[p1_pid_to_idx[npid]])
                tag = f"Step 1 rank={r}" + ("  <- deep in Step 1, new signal" if r > 10 else "")
            else:
                tag = "not in Step 1 catalogue"
            print(f"  {sim:.3f}  {name(npid):<45} [{tag}]")


Whole Milk (4210) — where each method's top-10 sits in Step 1's own ranking

[constant_theta]
  0.804  Fat Free Skim Milk                            [Step 1 rank=128  <- deep in Step 1, new signal]
  0.791  Skim Milk                                     [Step 1 rank=114  <- deep in Step 1, new signal]
  0.789  Milk, Fat Free                                [Step 1 rank=14  <- deep in Step 1, new signal]
  0.781  Goat Milk                                     [Step 1 rank=15  <- deep in Step 1, new signal]
  0.763  Lowfat Goat Milk                              [Step 1 rank=51  <- deep in Step 1, new signal]
  0.758  Milk, Organic, Fat Free                       [Step 1 rank=39  <- deep in Step 1, new signal]
  0.753  Organic Low Fat Milk                          [Step 1 rank=86  <- deep in Step 1, new signal]
  0.753  Milk Whole                                    [Step 1 rank=1]
  0.752  Vitamin D Organic Whole Milk                  [Step 1 rank=210  <- deep in Step 1, new signal]
  0.751